<a href="https://colab.research.google.com/github/192565027simats/CSA6102/blob/main/EXP40-Malware%20Static%20Analysis%20(Hashing%20and%20Entropy%20Check).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import hashlib
import math
import os
from collections import Counter


def sha256_hash(data: bytes) -> str:
    """
    Compute the SHA-256 hash of the given data.
    """
    return hashlib.sha256(data).hexdigest()


def byte_entropy(data: bytes) -> float:
    """
    Compute the Shannon entropy of a byte sequence.
    """
    if not data:
        return 0.0

    counts = Counter(data)
    length = len(data)

    return -sum(
        (count / length) * math.log2(count / length)
        for count in counts.values()
    )


def static_analyze(data: bytes, entropy_threshold=7.5):
    """
    Perform static analysis on binary data.
    """
    entropy = byte_entropy(data)

    return {
        "sha256": sha256_hash(data),
        "size_bytes": len(data),
        "entropy": round(entropy, 2),
        "likely_packed": entropy >= entropy_threshold,
    }


# ---------------- Sample Input ----------------

plain_sample = b"This is a normal configuration file. " * 20
random_sample = os.urandom(2000)

# ---------------- Run Analysis ----------------

plain_result = static_analyze(plain_sample)
random_result = static_analyze(random_sample)

# ---------------- Verification ----------------

assert len(plain_result["sha256"]) == 64
assert plain_result["entropy"] < 5.0
assert plain_result["likely_packed"] is False

assert random_result["entropy"] >= 7.5
assert random_result["likely_packed"] is True

assert sha256_hash(plain_sample) == sha256_hash(plain_sample)

print("All test cases passed.\n")

print("Normal File Analysis")
print("--------------------")
print(f"SHA-256       : {plain_result['sha256']}")
print(f"Size (bytes)  : {plain_result['size_bytes']}")
print(f"Entropy       : {plain_result['entropy']}")
print(f"Likely Packed : {plain_result['likely_packed']}")

print("\nRandom Binary Analysis")
print("----------------------")
print(f"SHA-256       : {random_result['sha256']}")
print(f"Size (bytes)  : {random_result['size_bytes']}")
print(f"Entropy       : {random_result['entropy']}")
print(f"Likely Packed : {random_result['likely_packed']}")

All test cases passed.

Normal File Analysis
--------------------
SHA-256       : 0ec14f14db669133737c6686d0ddf53a2f1f6096442e44420d38b218642c6f4b
Size (bytes)  : 740
Entropy       : 3.87
Likely Packed : False

Random Binary Analysis
----------------------
SHA-256       : 9ffb0039daedbec00ad547f9b3c47535f23e0492801fbfe344496a37cef4cde1
Size (bytes)  : 2000
Entropy       : 7.89
Likely Packed : True
